# Basic Dopaminergic Neuron Simulations

This notebook contains basic simulations of the dopaminergic (DA) neuron model, demonstrating the model's dynamics under various conditions.

---

# *This file allows to generate all plots necessary for figures*

# **Useful packages and functions**

## Dependencies and Setup

In [ ]:
using DifferentialEquations, Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("DA_kinetics.jl") # Loading of DA kinetics of gating variables
include("DA_models.jl") # Loading of DA model
include("DA_utils.jl"); # Loading of some utils functions

# **Global variables**

## Model Parameters

In [ ]:
# Definition of simulation time (in ms)
const Tfinal = 20000
const tspan  = (0.0, Tfinal)
tt = 0. : 0.01 : Tfinal
tt_rand = 0. : 1 : Tfinal

# Definition of reversal potential values (in mV), [Mg] and membrane capacitance
const VNa     = 60. # Sodium reversal potential
const VK      = -90. # Potassium reversal potential
const VCa     = 50. # Calcium reversal potential
const VH      = -29. # H reversal potential
const VLNS    = -65. # Leak reversal potential
const EPacemaker = 4.2732015978991615 # Reversal potential of pacemaking channels

const C       = 1. # Membrane capacitance
const fCa     = 0.018 # Fraction of unbuffered free calcium
const ICapmax = 11 # Maximum calcium pump current
const F       = 96520 # Faraday constant in ms*µA/mmol (and taking cm³=mL)
const d       = 15 # Soma diameter in cm
const L       = 25 # Soma length

# Definition of voltage range for the DICs
const Vmin = -100 
const Vmax = 50
const Vrange = range(Vmin, stop=Vmax, step=0.0154640);

## Plotting Configuration

In [ ]:
# Modifying backend GR attributes
gr(guidefontsize=25, tickfontsize=15, legendfontsize=12, margin=10Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1);
default(fmt = :png);

In [ ]:
# Define a struct (optional, but useful if you need parameters)
struct NoisyFunction
    amplitude::Float64  # amplitude of the noise
end

# Overload the () operator to make the struct callable
function (nf::NoisyFunction)(x::Float64)
    noise = nf.amplitude * randn()  # Generate Gaussian noise (mean 0, std 1)
    return noise  # Example function with noise
end

function condition(u,t,integrator) # Event when event_f(u,t) == 0
  (u[1]- (-20.))
end

function affect!(integrator)
end

cb = ContinuousCallback(condition, affect!, nothing, save_positions = (true, false));

# Simu base model

In [ ]:
gNa   = 6. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 1.117 # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.28 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa)

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE, x0, tspan, p) # Describing the problem
sol = solve(prob); # Solving the problem

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, 
    legend=false, ylims=(-100, 10), margins=20Plots.px, size=(1200, 800))
ylabel!("V (mV)")

current = plot(tt./1e3, Iapp.(tt), linewidth=2.5, color=myDarkBlue,
    legend=false, margins=20Plots.px, size=(1200, 800))
ylabel!("I (pA)")
xlabel!("t (s)")

l = @layout [
    a{1.0*w, 0.70*h}
    b{1.0*w, 0.30*h}
]
VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19, 20))
display(VI)

# **Figures**

In [ ]:
gNa   = 6. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 1.117 # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.28 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa)

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE, x0, tspan, p) # Describing the problem
sol = solve(prob); # Solving the problem

In [ ]:
sol_spike_times = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
chosen_spike = sol_spike_times.t[50];

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt, V_plot, linewidth=2.5, color=myDarkBlue, legend=false, ylims=(-90, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(chosen_spike-10, chosen_spike+10), 
    xticks=([chosen_spike-10, chosen_spike-5, chosen_spike, chosen_spike+5, chosen_spike+10], 
            ["0", "5", "10", "15", "20"]))
ylabel!("V (mV)")
xlabel!("t (ms)")
display(voltage)
# savefig(voltage, "./figures/fig1_single_spike.pdf")
# savefig(voltage, "./figures/fig1_single_spike.svg")

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, legend=false, ylims=(-70, 10), 
    margins=20Plots.px, size=(1200, 800), xlims=(17.6, 19.6), xticks=([17.6, 18.6, 19.6], 
            ["0", "1", "2"]))
ylabel!("V (mV)")
xlabel!("t (s)")
display(voltage)
# savefig(voltage, "./figures/fig1_full_trace.pdf")
# savefig(voltage, "./figures/fig1_full_trace.svg")

In [ ]:
p = plot(Vrange, m_inf.(Vrange), linewidth=2.5, label="Original model", color=myDarkBlue,
    legend=:topleft, ylims=(0, 1), margins=20Plots.px, size=(1200, 800), legendfontsize=20)
plot!(Vrange, m_inf_true.(Vrange), linewidth=2.5, label="Adjusted model", color=myGray)
plot!([-30, -30], [0, 0.5], linestyle=:dash, color=myDarkBlue, linewidth=2.5, label="")
plot!([-10, -10], [0, 0.5], linestyle=:dash, color=myGray, linewidth=2.5, label="")
ylabel!("Activation of transient Na channels")
xlabel!("V (mV)")
display(p)
# savefig(p, "./figures/fig1_activations.pdf")
# savefig(p, "./figures/fig1_activations.svg")

In [ ]:
p = plot(Vrange, l_inf.(Vrange), linewidth=2.5, label="Original model", color=myDarkBlue,
    legend=:topleft, ylims=(0, 1), margins=20Plots.px, size=(1200, 800), legendfontsize=20)
plot!(Vrange, l_inf_true.(Vrange), linewidth=2.5, label="Adjusted model", color=myGray)
plot!([-45, -45], [0, 0.5], linestyle=:dash, color=myDarkBlue, linewidth=2.5, label="")
plot!([-10, -10], [0, 0.5], linestyle=:dash, color=myGray, linewidth=2.5, label="")
ylabel!("Activation of CaL channels")
xlabel!("V (mV)")
display(p)
# savefig(p, "./figures/fig1_activationsCa.pdf")
# savefig(p, "./figures/fig1_activationsCa.svg")

In [ ]:
gNa   = 6. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 1.117 # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.28 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa)

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob_block = ODEProblem(DA_ODE, x0, tspan, p) # Describing the problem
sol_block = solve(prob_block); # Solving the problem

In [ ]:
# Retrieving variables
x         = sol_block(tt)
V_plot    = x[1, :]
voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, legend=false, ylims=(-70, 10), 
    margins=20Plots.px, size=(1200, 800), xlims=(17.6, 19.6), xticks=([17.6, 18.6, 19.6], 
            ["0", "1", "2"]))
ylabel!("V (mV)")
xlabel!("t (s)")
display(voltage)
# savefig(voltage, "./figures/fig1_full_trace_blockade.pdf")
# savefig(voltage, "./figures/fig1_full_trace_blockade.svg")

In [ ]:
tt_currentscapte = 19000 : 0.1 : 20000

In [ ]:
# Retrieving variables
x         = sol(tt_currentscapte)
V_plot    = x[1, :]
m_plot    = x[2, :]
h_plot    = x[3, :]
hs_plot   = x[4, :]
l_plot    = x[5, :]
n_plot    = x[6, :]
p_plot    = x[7, :]
q1_plot   = x[8, :]
q2_plot   = x[9, :]
o_plot    = x[10, :]
i_plot    = x[11, :]
mH_plot   = x[12, :]
Ca_plot   = x[13, :]

SK_inf_plot = zeros(length(tt_currentscapte))
for i = 1 : length(tt_currentscapte)
    if Ca_plot[i] > 0
        SK_inf_plot[i] = 1/(1+(0.00019/Ca_plot[i])^4)
    end
end

# Computing currents
INa = ((pi*d*L) / 100) .* gNa .* m_plot.^3 .* h_plot .* hs_plot .* (V_plot .- VNa)
ICaL = ((pi*d*L) / 100) .* gCaL .* l_plot .* (V_plot .- VCa)
IKd = ((pi*d*L) / 100) .* gKd .* n_plot.^3 .* (V_plot .- VK)
IKA = ((pi*d*L) / 100) .* gKA .* p_plot .* (q1_plot ./ 2 .+ q2_plot ./ 2) .* (V_plot .- VK)
IKERG = ((pi*d*L) / 100) .* gKERG .* o_plot .* (V_plot .- VK)
IKSK = ((pi*d*L) / 100) .* gKSK .* SK_inf_plot .* (V_plot .- VK)
IH = ((pi*d*L) / 100) .* gH .* mH_plot .* (V_plot .- VH)
ILCa = ((pi*d*L) / 100) .* gLCa .* (V_plot .- VCa)
ILNS = ((pi*d*L) / 100) .* gLNS .* (V_plot .- VLNS)

# Splitting inward and outward currents
INa_out = max.(INa, 0)
INa_in = min.(INa, 0)
ICaL_out = max.(ICaL, 0)
ICaL_in = min.(ICaL, 0)
IKd_out = max.(IKd, 0)
IKd_in = min.(IKd, 0)
IKA_out = max.(IKA, 0)
IKA_in = min.(IKA, 0)
IKERG_out = max.(IKERG, 0)
IKERG_in = min.(IKERG, 0)
IKSK_out = max.(IKSK, 0)
IKSK_in = min.(IKSK, 0)
IH_out = max.(IH, 0)
IH_in = min.(IH, 0)
ILCa_out = max.(ILCa, 0)
ILCa_in = min.(ILCa, 0)
ILNS_out = max.(ILNS, 0)
ILNS_in = min.(ILNS, 0)
I_in = INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ IKERG_in .+ IKSK_in .+ IH_in .+ ILCa_in .+ ILNS_in
I_out = INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ IKERG_out .+ IKSK_out .+ 
        IH_out .+ ILCa_out .+ ILNS_out;

In [ ]:
voltage = plot(tt_currentscapte./1e3, V_plot, linewidth=2.5, color=myDarkBlue, legend=false, ylims=(-70, 10), 
    margins=0Plots.px, size=(1200, 800), xaxis=false, xticks=false)
ylabel!("V (mV)")

outward_cur = plot(tt_currentscapte./1e3, I_out, linewidth=2.5, color=myDarkBlue, fillrange=zeros(length(tt_currentscapte)) .+ 1e-10,
    legend=false, margins=0Plots.px, size=(1200, 800), xaxis=false, xticks=false, 
    yscale=:log10, ylims=(1e1, 1e3), bott_currentscapteom_margin=-8Plots.px)
ylabel!("I out (pA)")

outward_cur_distr = plot(tt_currentscapte./1e3, zeros(length(tt_currentscapte)), linewidth=0, color=3, yticks=false,
    fillrange=100 .* (INa_out) ./ I_out, legend=:topright, margins=0Plots.px, size=(1200, 800), 
    xaxis=false, xticks=false, ylims=(0, 100), label="INa", 
    top_margin=-8Plots.px, bott_currentscapteom_margin=-8Plots.px)

plot!(tt_currentscapte./1e3, 100 .* (INa_out) ./ I_out, linewidth=0, color=2, 
    fillrange=100 .* (INa_out .+ ICaL_out) ./ I_out, label="ICaL")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out) ./ I_out, linewidth=0, color=1, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out) ./ I_out, label="IKd")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out) ./ I_out, linewidth=0, color=4, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out) ./ I_out, label="IKA")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out) ./ I_out, linewidth=0, color=5, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                      IKERG_out) ./ I_out, label="IKERG")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                       IKERG_out) ./ I_out, linewidth=0, color=7, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                      IKERG_out .+ IKSK_out) ./ I_out, label="IKSK")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                       IKERG_out .+ IKSK_out) ./ I_out, linewidth=0, color=6, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                      IKERG_out .+ IKSK_out .+ IH_out) ./ I_out, label="IH")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                       IKERG_out .+ IKSK_out .+ IH_out) ./ I_out, linewidth=0, color=8, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                      IKERG_out .+ IKSK_out .+ IH_out .+ ILCa_out) ./ I_out, label="ILCa")

plot!(tt_currentscapte./1e3, 100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                       IKERG_out .+ IKSK_out .+ IH_out .+ ILCa_out) ./ I_out, linewidth=0, color=myApple, 
    fillrange=100 .* (INa_out .+ ICaL_out .+ IKd_out .+ IKA_out .+ 
                      IKERG_out .+ IKSK_out .+ IH_out .+ ILCa_out .+ ILNS_out) ./ I_out, label="ILNS")
ylabel!("outward (%)")

inward_cur_distr = plot(tt_currentscapte./1e3, zeros(length(tt_currentscapte)), linewidth=0, color=3, yticks=false,
    fillrange=100 .* (INa_in) ./ I_in, legend=false, margins=0Plots.px, size=(1200, 800), 
    xaxis=false, xticks=false, ylims=(0, 100), label="INa", 
    top_margin=-8Plots.px, bott_currentscapteom_margin=-8Plots.px)

plot!(tt_currentscapte./1e3, 100 .* (INa_in) ./ I_in, linewidth=0, color=2, 
    fillrange=100 .* (INa_in .+ ICaL_in) ./ I_in, label="ICaL")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in) ./ I_in, linewidth=0, color=1, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in) ./ I_in, label="IKd")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in) ./ I_in, linewidth=0, color=4, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in) ./ I_in, label="IKA")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in) ./ I_in, linewidth=0, color=5, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                      IKERG_in) ./ I_in, label="IKERG")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                       IKERG_in) ./ I_in, linewidth=0, color=7, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                      IKERG_in .+ IKSK_in) ./ I_in, label="IKSK")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                       IKERG_in .+ IKSK_in) ./ I_in, linewidth=0, color=6, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                      IKERG_in .+ IKSK_in .+ IH_in) ./ I_in, label="IH")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                       IKERG_in .+ IKSK_in .+ IH_in) ./ I_in, linewidth=0, color=8, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                      IKERG_in .+ IKSK_in .+ IH_in .+ ILCa_in) ./ I_in, label="ILCa")

plot!(tt_currentscapte./1e3, 100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                       IKERG_in .+ IKSK_in .+ IH_in .+ ILCa_in) ./ I_in, linewidth=0, color=myApple, 
    fillrange=100 .* (INa_in .+ ICaL_in .+ IKd_in .+ IKA_in .+ 
                      IKERG_in .+ IKSK_in .+ IH_in .+ ILCa_in .+ ILNS_in) ./ I_in, label="ILNS")
ylabel!("inward (%)")

inward_cur = plot(tt_currentscapte./1e3, -I_in, linewidth=2.5, color=myDarkBlue, fillrange=zeros(length(tt_currentscapte)) .+ 1e-10,
    legend=false, margins=0Plots.px, size=(1200, 800), top_margin=-8Plots.px,
    yscale=:log10, ylims=(1e1, 1e3), yflip=true, xticks=([19, 20], ["0", "1"]))
ylabel!("I in (-pA)")
xlabel!("t (s)")

l = @layout [
    a{1.0*w, 0.2*h}
    b{1.0*w, 0.1*h}
    c{1.0*w, 0.3*h}
    d{1.0*w, 0.3*h}
    d{1.0*w, 0.1*h}
]
currentscape = plot(voltage, outward_cur, outward_cur_distr, inward_cur_distr, inward_cur, 
                    layout=l, size=(1000, 1500), xlims=(19, 20), left_margin=30Plots.px,
                    right_margin=0Plots.px)#, dpi=300)
display(currentscape)
# savefig(currentscape, "./figures/fig1_currentscape.png")
# savefig(currentscape, "./figures/fig1_currentscape.pdf")
# savefig(currentscape, "./figures/fig1_currentscape.svg")

# Figure 2

In [ ]:
gNa   = 6. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 1.117 # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.28 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_original = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa)

    # Simulation
    prob = ODEProblem(DA_ODE, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_original[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_original[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_original_model.dat", freqs_original);

In [ ]:
gNa   = 25. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 3. # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.025 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_adjusted = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, 0)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 100
        freqs_adjusted[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_adjusted[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_adjusted_model.dat", freqs_adjusted);

In [ ]:
Iapps = range(0, 20, length=100)
freqs_original = readdlm("./data/freqs_FI_original_model.dat")
freqs_adjusted = readdlm("./data/freqs_FI_adjusted_model.dat")
FI = plot(Iapps, freqs_original, seriestype=:scatter, legend=:topleft, linewidth=2.5, color=myDarkBlue, 
          size=(1200, 800), markersize=10, label="Original model", legendfontsize=20, margins=20Plots.px)
plot!(Iapps, freqs_adjusted, seriestype=:scatter, markersize=10, linewidth=2.5, color=myGray, 
      label="Adjusted model")
ylabel!("f (Hz)")
xlabel!("Applied current (pA)")
ylims!((0, 15))
display(FI)
# savefig(FI, "./figures/fig2_FIs.pdf")
# savefig(FI, "./figures/fig2_FIs.svg")

In [ ]:
gNa   = 25. # Sodium current maximal conductance
gCaL  = 0.139 # L-type calcium current maximal conductance
gKd   = 3. # Delayed-rectifier potassium current maximal conductance
gKA   = 1.68 # A-type potassium current maximal conductance
gKERG = 0.13 # ERG current maximal conductance
gKSK  = 0.07 # SK current maximal conductance
gH    = 0.078 # H current maximal conductance
gLNS  = 0.025 # Leak non specific current maximal conductance
gLCa  = 0.00245 # Leak calcium current maximal conductance

# Input current definition
Iapp(t) = 15 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, 0)

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
sol = solve(prob); # Solving the problem

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-70, 10), 
    margins=20Plots.px, size=(1200, 800), xlims=(19, 20), xticks=([19, 19.5, 20], 
            ["0", "0.5", "1"]))
ylabel!("V (mV)")
xlabel!("t (s)")
display(voltage)
# savefig(voltage, "./figures/fig2_full_trace.pdf")
# savefig(voltage, "./figures/fig2_full_trace.svg")

In [ ]:
gNa     = 25. # Sodium current maximal conductance
gCaL    = 0.05 # L-type calcium current maximal conductance
gKd     = 3. # Delayed-rectifier potassium current maximal conductance
gKA     = 1.68 # A-type potassium current maximal conductance
gKERG   = 0.13 # ERG current maximal conductance
gKSK    = 0.07 # SK current maximal conductance
gH      = 0.078 # H current maximal conductance
gLNS    = 0.011 # Leak non specific current maximal conductance
gLCa    = 0.00245 # Leak calcium current maximal conductance
gNaLCNs = range(0, 0.025, length=100)

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapp(t) = 0 # pA

# Initializing variables
freqs_NaLCN = zeros(length(gNaLCNs))

for (i, gNaLCN) in enumerate(gNaLCNs)
    display(i)
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gNaLCN)

    # Simulation
    prob = ODEProblem(DA_ODE_true_NaLCN, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_NaLCN[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_NaLCN[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_NaLCN.dat", freqs_NaLCN);

In [ ]:
freqs_NaLCN = readdlm("./data/freqs_NaLCN.dat")
FNaLCN = plot(gNaLCNs, freqs_NaLCN, seriestype=:scatter, legend=false, linewidth=2.5, color=myRed, 
          size=(1200, 800), markersize=10, xlims=(0, 0.025), margins=25Plots.px)
ylabel!("f (Hz)")
xlabel!("NaLCN conductance (mS/cm²)")
ylims!((0, 25))
display(FNaLCN)
# savefig(FNaLCN, "./figures/fig2_FNaLCN.pdf")
# savefig(FNaLCN, "./figures/fig2_FNaLCN.svg")

In [ ]:
gNa     = 25. # Sodium current maximal conductance
gCaL    = 0.05 # L-type calcium current maximal conductance
gKd     = 3. # Delayed-rectifier potassium current maximal conductance
gKA     = 1.68 # A-type potassium current maximal conductance
gKERG   = 0.13 # ERG current maximal conductance
gKSK    = 0.07 # SK current maximal conductance
gH      = 0.078 # H current maximal conductance
gLNS    = 0.011 # Leak non specific current maximal conductance
gLCa    = 0.00245 # Leak calcium current maximal conductance
gNaLCN  = 0.012

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gNaLCN)

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_NaLCN, x0, tspan, p) # Describing the problem
sol = solve(prob; maxiters=1e6); # Solving the problem

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myRed, legend=false, ylims=(-90, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(18.9, 19.9), xticks=([18.9, 19.4, 19.9], 
            ["0", "0.5", "1"]))
ylabel!("V (mV)")
xlabel!("t (s)")
display(voltage)
# savefig(voltage, "./figures/fig2_full_trace_NaLCN.pdf")
# savefig(voltage, "./figures/fig2_full_trace_NaLCN.svg")

In [ ]:
gNa     = 25. # Sodium current maximal conductance
gCaL    = 0.05 # L-type calcium current maximal conductance
gKd     = 3. # Delayed-rectifier potassium current maximal conductance
gKA     = 1.68 # A-type potassium current maximal conductance
gKERG   = 0.13 # ERG current maximal conductance
gKSK    = 0.07 # SK current maximal conductance
gH      = 0.078 # H current maximal conductance
gLNS    = 0.011 # Leak non specific current maximal conductance
gLCa    = 0.00245 # Leak calcium current maximal conductance
gNaLCN  = 0.004

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_FI_NaLCN = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gNaLCN)

    # Simulation
    prob = ODEProblem(DA_ODE_true_NaLCN, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 100
        freqs_FI_NaLCN[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_FI_NaLCN[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_NaLCN.dat", freqs_FI_NaLCN);

In [ ]:
Iapps = range(0, 20, length=100)
freqs_FI_NaLCN = readdlm("./data/freqs_FI_NaLCN.dat")
FI = plot(Iapps, freqs_FI_NaLCN, seriestype=:scatter, legend=:topleft, linewidth=2.5, color=myRed, 
          size=(1200, 800), markersize=10, label=false, legendfontsize=20, margins=20Plots.px)
ylabel!("f (Hz)")
xlabel!("Applied current (pA)")
ylims!((0, 25))
display(FI)
# savefig(FI, "./figures/fig2_FI_NaLCN.pdf")
# savefig(FI, "./figures/fig2_FI_NaLCN.svg")

In [ ]:
gNa     = 25. # Sodium current maximal conductance
gCaL    = 0.05 # L-type calcium current maximal conductance
gKd     = 3. # Delayed-rectifier potassium current maximal conductance
gKA     = 1.68 # A-type potassium current maximal conductance
gKERG   = 0.13 # ERG current maximal conductance
gKSK    = 0.07 # SK current maximal conductance
gH      = 0.078 # H current maximal conductance
gLNS    = 0.011 # Leak non specific current maximal conductance
gLCa    = 0.00245 # Leak calcium current maximal conductance
gNaLCN  = 0.012

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_FI_NaLCN2 = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gNaLCN)

    # Simulation
    prob = ODEProblem(DA_ODE_true_NaLCN, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 100
        freqs_FI_NaLCN2[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_FI_NaLCN2[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_NaLCN2.dat", freqs_FI_NaLCN2);

In [ ]:
Iapps = range(0, 20, length=100)
freqs_FI_NaLCN2 = readdlm("./data/freqs_FI_NaLCN2.dat")
FI = plot(Iapps, freqs_FI_NaLCN2, seriestype=:scatter, legend=:topleft, linewidth=2.5, color=myRed, 
          size=(1200, 800), markersize=10, label=false, legendfontsize=20, margins=20Plots.px)
ylabel!("f (Hz)")
xlabel!("Applied current (pA)")
ylims!((0, 30))
display(FI)
# savefig(FI, "./figures/fig2_FI_NaLCN2.pdf")
# savefig(FI, "./figures/fig2_FI_NaLCN2.svg")